# Pathologist Viewing-Behaviour — Exploratory Insight Analysis

Two near-complete readers (`muhammad.aslam` 250 slides, `ashish.bansal` 185) grading
colorectal slides (IMP-CRS-2024: non-neoplastic / low-grade / high-grade) in the
constrained click-to-zoom study tool. Source: `analysis/data/pathology_events_2026-06-16.csv`.

Three threads: **(1) label & convention structure**, **(2) navigation & trajectory**,
**(3) behaviour ↔ diagnosis bridge** — then a synthesis. The notebook *asserts* the
hand-computed anchor numbers so any pipeline drift fails loudly.

Caveat carried throughout: viewport/cursor position is a *constrained-navigation
attention proxy*, **not** eye-tracking; there are 2 readers (not 5) and no per-region
ground truth.

In [1]:
import os, sys, json
import numpy as np
import pandas as pd

try:  # robust unicode on the Windows console (cp1252) and under nbconvert
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    HERE = os.path.abspath(".")
ANALYSIS = os.path.dirname(HERE)
SRC = os.path.join(ANALYSIS, "src")
# The CSV is gitignored (repo convention: *.csv data exports stay local). Look in
# analysis/data/ first, then fall back to the repo-root export.
_DATA_CANDIDATES = [
    os.path.join(ANALYSIS, "data", "pathology_events_2026-06-16.csv"),
    os.path.join(os.path.dirname(ANALYSIS), "pathology_events_2026-06-16.csv"),
]
DATA = next((p for p in _DATA_CANDIDATES if os.path.exists(p)), _DATA_CANDIDATES[0])
if not os.path.exists(DATA):
    raise FileNotFoundError(
        "pathology_events_2026-06-16.csv not found (gitignored per repo convention). "
        "Place it in analysis/data/. Tried: " + " ; ".join(_DATA_CANDIDATES))
FIGDIR = os.path.join(ANALYSIS, "figures")
REPORTS = os.path.join(ANALYSIS, "reports")
sys.path.insert(0, SRC)

import load, labels, metrics, navigation, figures
from load import TWO_READERS, GRADE_LABELS

READERS = TWO_READERS
results = {}  # accumulates everything for the report / gbrain page

## 0. Load & descriptive overview

In [2]:
ev_all = load.load_events(DATA)
print("total events:", len(ev_all))
by_user = (ev_all.groupby("user_id")
           .agg(events=("event", "size"), slides=("slide_id", "nunique"))
           .sort_values("events", ascending=False))
print(by_user)
ev = load.reader_events(ev_all, READERS)

results["overview"] = {
    "total_events": int(len(ev_all)),
    "by_user": {u: {"events": int(r.events), "slides": int(r.slides)}
                for u, r in by_user.iterrows()},
    "event_type_counts": ev_all["event"].value_counts().to_dict(),
    "ground_truth_event_counts": ev_all["ground_truth"].value_counts().to_dict(),
}

total events: 17354
                events  slides
user_id                       
muhammad.aslam    8341     250
ashish.bansal     7152     185
iftikhar.rana     1124      11
alistair.heath     503       8
admin              234       2


## Thread 1 — Label & convention structure  *(lead; serves B1)*

Event-level committed diagnoses (every `slide_next` with a label) give the headline
accuracy/confusion; this is the anchor granularity.

In [3]:
thread1 = {}
confusions, accuracies = {}, {}
for r in READERS:
    diag = labels.slide_next_diagnoses(ev[ev["user_id"] == r])
    cm = labels.confusion(diag)
    acc = float(diag["correct"].mean())
    confusions[r] = cm
    accuracies[r] = acc
    high_under = int(cm.loc["high-grade", "low-grade"])
    high_total = int(cm.loc["high-grade"].sum())
    thread1[r] = {
        "n_committed": int(len(diag)),
        "accuracy": acc,
        "balanced_accuracy": metrics.balanced_accuracy(cm),
        "per_class_recall": metrics.per_class_recall(cm).to_dict(),
        "high_to_low_undergrade": [high_under, high_total],
        "mean_signed_error": float(diag["signed_error"].mean()),
        "direction_counts": diag["direction"].value_counts().to_dict(),
        "confusion": cm.to_dict(),
    }
    print(f"\n=== {r} ===  n={len(diag)}  acc={acc:.1%}  "
          f"high→low {high_under}/{high_total}  mean signed err={diag['signed_error'].mean():+.2f}")
    print(cm)
results["thread1_per_reader"] = thread1


=== muhammad.aslam ===  n=236  acc=64.8%  high→low 48/78  mean signed err=-0.18
label           non-neoplastic  low-grade  high-grade
ground_truth                                         
non-neoplastic              51         16           0
low-grade                   15         72           4
high-grade                   0         48          30

=== ashish.bansal ===  n=162  acc=69.1%  high→low 29/51  mean signed err=-0.22
label           non-neoplastic  low-grade  high-grade
ground_truth                                         
non-neoplastic              39          6           0
low-grade                   14         51           1
high-grade                   0         29          22


### Inter-rater: do the readers agree with *each other* more than with the labels?
Slide-level (last committed diagnosis per (reader, slide)), `slide_next` only — anchor granularity.

In [4]:
slide_diag = metrics.slide_level_diagnoses(ev, use_fallback=False)
ir = metrics.interrater_summary(slide_diag, READERS[0], READERS[1])
print(json.dumps({k: v for k, v in ir.items()}, indent=2, default=float))
results["interrater"] = ir

# fallback version (recovers label_select-only views) for transparency
slide_diag_fb = metrics.slide_level_diagnoses(ev, use_fallback=True)
ir_fb = metrics.interrater_summary(slide_diag_fb, READERS[0], READERS[1])
results["interrater_with_fallback"] = ir_fb

{
  "n_shared": 151,
  "agree_ab": 133,
  "agree_ab_rate": 0.8807947019867549,
  "both_correct": 91,
  "both_correct_rate": 0.6026490066225165,
  "both_wrong_same": 42,
  "one_correct": 18,
  "n_high_shared": 47,
  "both_under_high": 25,
  "kappa_ab": 0.8149257796540923,
  "kappa_a_gt": 0.5167999999999999,
  "kappa_b_gt": 0.611129772701825,
  "kappa_ab_unweighted": 0.790310137324487,
  "kappa_a_gt_unweighted": 0.42157466310965186,
  "kappa_b_gt_unweighted": 0.5280929410965418
}


**Shared-bias vs noise.** Of the disagreements-with-GT on shared slides, how many are
*concordant* (both readers make the same wrong call = shared convention) vs idiosyncratic?

In [5]:
disagree = ir["n_shared"] - ir["both_correct"]
concordant_error = ir["both_wrong_same"]
print(f"shared slides={ir['n_shared']}  both-correct={ir['both_correct']} "
      f"({ir['both_correct_rate']:.0%})  readers-agree={ir['agree_ab']} ({ir['agree_ab_rate']:.0%})")
print(f"of {disagree} GT-disagreements, {concordant_error} are CONCORDANT (shared bias), "
      f"{ir['one_correct']} split")
print(f"high-grade shared slides={ir['n_high_shared']}, BOTH undergraded high→low={ir['both_under_high']}")
results["shared_bias"] = {
    "n_gt_disagreements": int(disagree),
    "concordant_errors": int(concordant_error),
    "split_errors": int(ir["one_correct"]),
    "concordant_fraction_of_errors": float(concordant_error / disagree) if disagree else None,
}

shared slides=151  both-correct=91 (60%)  readers-agree=133 (88%)
of 60 GT-disagreements, 42 are CONCORDANT (shared bias), 18 split
high-grade shared slides=47, BOTH undergraded high→low=25


### Figures 1–3

In [6]:
print(figures.fig_confusion(confusions, accuracies, FIGDIR))
print(figures.fig_signed_error(slide_diag, READERS, FIGDIR))
print(figures.fig_agreement(ir, READERS, FIGDIR))

C:\Users\conno\Documents\GitHub\pathologist_user_study\.claude\worktrees\happy-williamson-e78e35\analysis\figures\01_confusion.png


C:\Users\conno\Documents\GitHub\pathologist_user_study\.claude\worktrees\happy-williamson-e78e35\analysis\figures\02_signed_error.png


C:\Users\conno\Documents\GitHub\pathologist_user_study\.claude\worktrees\happy-williamson-e78e35\analysis\figures\03_agreement.png


## Thread 2 — Navigation & trajectory  *(serves B2)*
Build the per-slide-view feature table (behaviour + diagnosis).

In [7]:
feat = navigation.build_feature_table(ev)
print("slide-views:", len(feat))
print(feat.groupby("user_id")[["active_dwell_s", "raw_dwell_s", "n_clicks",
                               "n_zoom_step", "max_zoom"]].median())

results["behaviour_medians"] = {
    r: feat[feat["user_id"] == r][["active_dwell_s", "raw_dwell_s", "n_clicks",
                                   "n_zoom_step", "max_zoom", "frac_time_ge_10x"]].median().to_dict()
    for r in READERS
}

slide-views: 441
                active_dwell_s  raw_dwell_s  n_clicks  n_zoom_step  max_zoom
user_id                                                                     
ashish.bansal             29.0         29.0       5.0         10.0       5.0
muhammad.aslam            18.0         18.0       5.0         10.0       5.0


### Coarse-to-fine: do views zoom *in* over time ("rule-out then confirm")?

In [8]:
ctf = feat["ctf_spearman"].dropna()
from scipy.stats import wilcoxon
ctf_stat = {
    "mean_rho": float(ctf.mean()),
    "median_rho": float(ctf.median()),
    "frac_positive": float((ctf > 0).mean()),
    "n_views_scored": int(len(ctf)),
    "wilcoxon_p_vs_0": float(wilcoxon(ctf)[1]) if len(ctf) > 10 else None,
}
print("coarse-to-fine:", ctf_stat)
results["coarse_to_fine"] = ctf_stat

coarse-to-fine: {'mean_rho': 0.4491980941082164, 'median_rho': 0.4581325443926032, 'frac_positive': 0.9491916859122402, 'n_views_scored': 433, 'wilcoxon_p_vs_0': 2.2032348681581365e-68}


### B2 shuffle falsifier — does navigation *order* carry signal?

In [9]:
shuf = metrics.shuffle_falsifier(ev, n_shuffles=200, seed=0)
print({k: v for k, v in shuf.items() if k != "h_shuffled_samples"})
results["shuffle_falsifier"] = {k: v for k, v in shuf.items() if k != "h_shuffled_samples"}

{'h_real_bits': 1.0049750214415878, 'h_shuffled_mean_bits': 2.089856289431934, 'h_shuffled_std_bits': 0.0070545307814240885, 'entropy_reduction_bits': 1.0848812679903463, 'p_value': 0.0, 'n_views': 433}


### Figures 4–5, 7 (zoom usage, example scanpaths, shuffle)

In [10]:
tmag = navigation.time_at_magnification_table(ev)
print(figures.fig_zoom_usage(feat, tmag, READERS, FIGDIR))

# pick scanpath examples: a high-grade slide undergraded vs correctly graded (most viewport points)
def _example_views():
    out = []
    hg = feat[(feat["ground_truth"] == "high-grade") & feat["committed_label"].notna()].copy()
    hg = hg.sort_values("n_viewport_poll", ascending=False)
    under = hg[hg["committed_label"] == "low-grade"].head(1)
    corr = hg[hg["committed_label"] == "high-grade"].head(1)
    for tag, row in [("high-grade slide read as LOW", under), ("high-grade read CORRECTLY", corr)]:
        if len(row):
            rr = row.iloc[0]
            g = ev[(ev["session_id"] == rr["session_id"]) &
                   (ev["viewing_attempt"] == rr["viewing_attempt"])]
            out.append((f"{rr['user_id']} · {rr['slide_id']}\n{tag}\n"
                        f"{rr['active_dwell_s']:.0f}s, max {rr['max_zoom']:g}×", g))
    return out

print(figures.fig_scanpaths(_example_views(), FIGDIR))
print(figures.fig_shuffle(shuf, FIGDIR))

C:\Users\conno\Documents\GitHub\pathologist_user_study\.claude\worktrees\happy-williamson-e78e35\analysis\figures\04_zoom_usage.png
C:\Users\conno\Documents\GitHub\pathologist_user_study\.claude\worktrees\happy-williamson-e78e35\analysis\figures\05_scanpaths.png
C:\Users\conno\Documents\GitHub\pathologist_user_study\.claude\worktrees\happy-williamson-e78e35\analysis\figures\07_shuffle_falsifier.png


## Thread 3 — Behaviour ↔ diagnosis bridge

Lead hypothesis: on high-grade slides, *under-graded* views got **less** examination
(lower zoom, shorter active-dwell, fewer clicks) than correctly-graded ones.

In [11]:
hg = feat[(feat["ground_truth"] == "high-grade") & feat["committed_label"].notna()].copy()
hg_under = hg[hg["committed_label"] == "low-grade"]
hg_corr = hg[hg["committed_label"] == "high-grade"]
bridge = {}
for col in ["active_dwell_s", "max_zoom", "n_clicks", "n_zoom_step", "frac_time_ge_10x", "n_fixations"]:
    bridge[col] = metrics.mann_whitney(hg_corr[col], hg_under[col])
print("High-grade: correct vs under-grade (Mann–Whitney, Cliff's δ):")
for k, v in bridge.items():
    print(f"  {k:18s} median corr={v['median_x']:.2f}  under={v['median_y']:.2f}  "
          f"δ={v['cliffs_delta']:+.2f}  p={v['p']:.3f}")
results["bridge_highgrade_correct_vs_under"] = bridge

High-grade: correct vs under-grade (Mann–Whitney, Cliff's δ):
  active_dwell_s     median corr=22.00  under=22.50  δ=-0.09  p=0.354
  max_zoom           median corr=5.00  under=5.00  δ=-0.00  p=0.966
  n_clicks           median corr=4.00  under=5.00  δ=-0.11  p=0.236
  n_zoom_step        median corr=7.00  under=10.50  δ=-0.19  p=0.049
  frac_time_ge_10x   median corr=0.00  under=0.00  δ=-0.00  p=0.978
  n_fixations        median corr=3.00  under=4.00  δ=-0.21  p=0.027


### Descriptive logistic model: predict an *error* from behaviour features

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

model_cols = ["active_dwell_s", "max_zoom", "n_clicks", "n_zoom_step",
              "frac_time_ge_10x", "path_len_px", "n_fixations", "label_switches"]
md = feat[feat["committed_label"].notna()].dropna(subset=model_cols + ["correct"]).copy()
X = StandardScaler().fit_transform(md[model_cols].values)
y = (~md["correct"].astype(bool)).astype(int).values  # 1 = error
clf = LogisticRegression(max_iter=1000).fit(X, y)
# bootstrap CIs
rng = np.random.default_rng(0)
boot = []
for _ in range(500):
    idx = rng.integers(0, len(y), len(y))
    if len(np.unique(y[idx])) < 2:
        continue
    boot.append(LogisticRegression(max_iter=1000).fit(X[idx], y[idx]).coef_[0])
boot = np.array(boot)
coef_tbl = pd.DataFrame({
    "feature": model_cols,
    "coef(std)": clf.coef_[0],
    "ci_lo": np.percentile(boot, 2.5, axis=0),
    "ci_hi": np.percentile(boot, 97.5, axis=0),
}).sort_values("coef(std)")
print(f"error-prediction logistic (n={len(y)}, errors={int(y.sum())}):")
print(coef_tbl.to_string(index=False))
results["error_logistic"] = {
    "n": int(len(y)), "n_errors": int(y.sum()),
    "coefficients": coef_tbl.to_dict(orient="records"),
}

error-prediction logistic (n=434, errors=146):
         feature  coef(std)     ci_lo    ci_hi
     path_len_px  -0.301592 -0.693376 0.088754
  label_switches  -0.084855 -0.593885 0.108617
     n_zoom_step  -0.014627 -0.558872 0.644194
        n_clicks   0.022425 -0.808344 0.513835
frac_time_ge_10x   0.054831 -0.240486 0.280797
        max_zoom   0.063200 -0.273092 0.377441
     n_fixations   0.187118 -0.112334 0.504877
  active_dwell_s   0.269490  0.005204 0.598337


### Label-switching as an uncertainty signal

In [13]:
lab = feat[feat["committed_label"].notna()].copy()
lab["switched"] = lab["label_switches"] > 1
sw = lab.groupby("switched")["correct"].agg(["mean", "size"])
print("error rate by mind-change:\n", (1 - sw["mean"]).rename("error_rate"))
results["label_switching"] = {
    "n_switched": int(lab["switched"].sum()),
    "error_rate_switched": float(1 - sw["mean"].get(True, np.nan)),
    "error_rate_settled": float(1 - sw["mean"].get(False, np.nan)),
}

error rate by mind-change:
 switched
False    0.338824
True     0.222222
Name: error_rate, dtype: object


### Speed vs accuracy, and revisits

In [14]:
speed = {}
for r in READERS:
    d = feat[(feat["user_id"] == r) & feat["committed_label"].notna()]
    speed[r] = metrics.mann_whitney(d[d["correct"] == True]["active_dwell_s"],
                                    d[d["correct"] == False]["active_dwell_s"])
    print(f"{r}: active-dwell correct vs error  "
          f"median {speed[r]['median_x']:.0f}s vs {speed[r]['median_y']:.0f}s  "
          f"δ={speed[r]['cliffs_delta']:+.2f} p={speed[r]['p']:.3f}")
results["speed_accuracy"] = speed

revisits = feat[feat["revisit"]]
rv = {"n_revisit_views": int(len(revisits)),
      "n_slides_revisited": int(revisits["slide_id"].nunique()),
      "revisit_accuracy": float(revisits["correct"].dropna().mean()) if len(revisits) else None}
print("revisits:", rv)
results["revisits"] = rv

# Figure 6 (behaviour by outcome)
print(figures.fig_behaviour_by_outcome(feat, FIGDIR))

muhammad.aslam: active-dwell correct vs error  median 17s vs 21s  δ=-0.19 p=0.015
ashish.bansal: active-dwell correct vs error  median 28s vs 29s  δ=-0.06 p=0.534
revisits: {'n_revisit_views': 8, 'n_slides_revisited': 7, 'revisit_accuracy': 0.42857142857142855}


C:\Users\conno\Documents\GitHub\pathologist_user_study\.claude\worktrees\happy-williamson-e78e35\analysis\figures\06_behaviour_by_outcome.png


## Anchor checks — fail loudly if the pipeline drifts

In [15]:
def approx(a, b, tol):  # noqa
    assert abs(a - b) <= tol, f"ANCHOR FAIL: {a} vs {b} (tol {tol})"

approx(accuracies["muhammad.aslam"], 0.648, 0.01)
approx(accuracies["ashish.bansal"], 0.691, 0.01)
assert thread1["muhammad.aslam"]["n_committed"] == 236, thread1["muhammad.aslam"]["n_committed"]
assert thread1["ashish.bansal"]["n_committed"] == 162, thread1["ashish.bansal"]["n_committed"]
assert thread1["muhammad.aslam"]["high_to_low_undergrade"] == [48, 78]
assert thread1["ashish.bansal"]["high_to_low_undergrade"] == [29, 51]
assert ir["n_shared"] == 151, ir["n_shared"]
assert ir["agree_ab"] == 133, ir["agree_ab"]
assert ir["both_correct"] == 91, ir["both_correct"]
assert ir["both_wrong_same"] == 42, ir["both_wrong_same"]
assert ir["n_high_shared"] == 47 and ir["both_under_high"] == 25
print("ALL ANCHORS OK ✓")

ALL ANCHORS OK ✓


## Persist results for the report / gbrain page

In [16]:
os.makedirs(REPORTS, exist_ok=True)
with open(os.path.join(REPORTS, "results.json"), "w") as f:
    json.dump(results, f, indent=2, default=lambda o: float(o) if isinstance(o, (np.floating,)) else int(o)
              if isinstance(o, (np.integer,)) else str(o))
print("wrote", os.path.join(REPORTS, "results.json"))

wrote C:\Users\conno\Documents\GitHub\pathologist_user_study\.claude\worktrees\happy-williamson-e78e35\analysis\reports\results.json
